# Notebook 02: LLM vs Rule-Based Controller Comparison

This notebook analyses the results of the full benchmark run to answer the primary research question:
> Does an LLM controller that reads region-level diagnostics and proposes targeted synonymous edits reach a better Pareto front than non-LLM optimizers under the same tool-call budget?

It assumes the benchmark was run using a command similar to:
```bash
mrna-design bench --controllers random,rule_based,llm --iters 100 --pop 10 --seeds 5 --out ../results/bench
```

We will load the `summary.json`, plot the hypervolume trajectories across all seeds, and run statistical tests.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mrna_design.eval.stats import compare_hypervolumes

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Imports OK')

## 1. Load Benchmark Results

We first look for the full benchmark results in `../results/bench/summary.json`. If not found, we fall back to a dry-run directory for demonstration purposes.

In [ ]:
bench_dir = Path("../results/bench")
if not bench_dir.exists():
    print(f"Full benchmark not found at {bench_dir}, falling back to dry_run")
    bench_dir = Path("../results/bench_dry_run")

summary_path = bench_dir / "summary.json"
with open(summary_path) as f:
    summary = json.load(f)

df = pd.DataFrame([s for s in summary if "error" not in s])
print(f"Loaded {len(df)} successful benchmark runs.")
df.head()

## 2. Statistical Analysis (Mann-Whitney U Test)

We compare the final hypervolumes reached by `llm` and `rule_based` against the `random` baseline.

In [ ]:
# Filter out any runs with 0 variance or too few seeds for meaningful stats if needed
stats_res = compare_hypervolumes(summary, baseline_controller="random")
print(json.dumps(stats_res, indent=2))

## 3. Plot Hypervolume Trajectories

We plot the mean hypervolume across iterations, with shaded regions representing the 95% confidence intervals across the different random seeds.

In [ ]:
targets = df['target'].unique()
fig, axes = plt.subplots(1, len(targets), figsize=(6 * len(targets), 5), sharey=True)
if len(targets) == 1:
    axes = [axes]

for ax, target in zip(axes, targets):
    target_data = [s for s in summary if s.get('target') == target and 'error' not in s]
    controllers = sorted(list(set(s['controller'] for s in target_data)))
    
    for ctrl in controllers:
        ctrl_runs = [s for s in target_data if s['controller'] == ctrl]
        # Extract step-by-step histories
        histories = []
        for run in ctrl_runs:
            iters = [h['iteration'] for h in run['hv_history']]
            hvs = [h['hv'] for h in run['hv_history']]
            histories.append(pd.Series(hvs, index=iters))
        
        if not histories:
            continue
            
        hist_df = pd.concat(histories, axis=1).ffill().fillna(0)
        mean_hv = hist_df.mean(axis=1)
        std_hv = hist_df.std(axis=1)
        
        ax.plot(mean_hv.index, mean_hv.values, label=ctrl, linewidth=2)
        ax.fill_between(mean_hv.index, mean_hv - std_hv, mean_hv + std_hv, alpha=0.2)
        
    ax.set_title(f"{target} Hypervolume", fontsize=13, fontweight='bold')
    ax.set_xlabel("Iteration")
    if ax == axes[0]:
        ax.set_ylabel("Hypervolume")
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Conclusion

By observing the effect sizes and P-values, we can determine whether the targeted LLM proposals significantly outperform rule-based edits when both operate under the same tool-call budget. The plots above illustrate the rate of convergence.